# Lab 08: DDPM on MNIST

**Module 08** | Read `notes/08-ddpm-stable-diffusion.pdf` before running this notebook.

Train a small noise-predictor U-Net on MNIST and sample images by reversing the diffusion process.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## DDPM on MNIST

A Denoising Diffusion Probabilistic Model (DDPM) trains a neural network to predict the noise `epsilon` that was added at each forward step. At sampling time we start from pure Gaussian noise and iteratively denoise using the learned reverse kernel with the same beta schedule from Lab 07, run backward.

This lab uses a compact U-Net as the noise predictor. The architecture is intentionally small so a few epochs on a MNIST subset finish quickly on CPU or GPU.


### Step 1: Schedule, data subset, and diffusion helpers

We reuse a linear beta schedule with `T = 200` timesteps (fewer than Lab 07 for speed). Training samples a random timestep per image, corrupts the batch with the closed-form forward formula, and minimizes MSE between true and predicted noise.

To keep runtime low we train on 8,000 images for three epochs.


In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "datasets" / "mnist"
T = 200
BATCH_SIZE = 128
EPOCHS = 3
LR = 2e-4
SUBSET_SIZE = 8000

betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
full_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
train_set = Subset(full_set, list(range(SUBSET_SIZE)))
loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print("DDPM training setup")
print(f"  Full MNIST train:  {len(full_set):,} images")
print(f"  Subset used:       {len(train_set):,} images (indices 0..{SUBSET_SIZE - 1})")
print(f"  Batch size:        {BATCH_SIZE}")
print(f"  Batches per epoch: {len(loader)}")
print(f"  Timesteps T:       {T}")
print(f"  Epochs:            {EPOCHS}")
print(f"  beta_1 / beta_T:   {betas[0]:.6f} / {betas[-1]:.4f}")
print(f"  alpha_bar_T:       {alpha_bars[-1]:.6f}")


### Step 2: Define the noise-predictor U-Net

The U-Net embeds the diffusion timestep with sinusoidal features and adds them as a bias after the first convolution. An encoder shrinks spatial resolution; a decoder upsamples with skip connections so fine digit strokes are preserved.

`GroupNorm` replaces batch norm, a common choice in diffusion models where batch statistics can be noisy with small batches.


In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        self.proj = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / half
        )
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2:
            emb = F.pad(emb, (0, 1))
        return self.proj(emb)


class ResBlock(nn.Module):
    def __init__(self, channels: int, time_dim: int):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.time = nn.Linear(time_dim, channels)
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = self.conv2(F.silu(self.norm2(h)))
        return x + h


class SimpleUNet(nn.Module):
    def __init__(self, time_dim: int = 64):
        super().__init__()
        self.time_emb = SinusoidalTimeEmbedding(time_dim)
        self.in_conv = nn.Conv2d(1, 32, 3, padding=1)
        self.down1 = ResBlock(32, time_dim)
        self.down2 = ResBlock(32, time_dim)
        self.pool = nn.MaxPool2d(2)
        self.mid = ResBlock(64, time_dim)
        self.up_conv = nn.Conv2d(64, 32, 3, padding=1)
        self.up1 = ResBlock(32, time_dim)
        self.up2 = ResBlock(32, time_dim)
        self.out_conv = nn.Conv2d(32, 1, 1)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_emb(t)
        h0 = self.in_conv(x)
        h1 = self.down1(h0, t_emb)
        h2 = self.down2(h1, t_emb)
        h = self.pool(h2)
        h = torch.cat([h, self.pool(h1)], dim=1)
        h = self.mid(h, t_emb)
        h = F.interpolate(h, scale_factor=2, mode="nearest")
        h = self.up_conv(h) + h2
        h = self.up1(h, t_emb)
        h = self.up2(h, t_emb)
        return self.out_conv(h)


def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
    sa = sqrt_alpha_bars[t].view(-1, 1, 1, 1).to(x0.device)
    so = sqrt_one_minus_alpha_bars[t].view(-1, 1, 1, 1).to(x0.device)
    return sa * x0 + so * noise


model = SimpleUNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
print(f"Noise predictor parameters: {sum(p.numel() for p in model.parameters()):,}")


### Step 3: Walk through one training batch

Before the full loop, we inspect a single batch: sample random timesteps `t`, add noise with `q_sample`, predict noise, and compute MSE for one example. This mirrors the inner training step and shows how loss scales with corruption level.


In [ ]:
torch.manual_seed(0)
demo_images, _ = next(iter(loader))
demo_images = demo_images.to(device)
demo_noise = torch.randn_like(demo_images)
demo_t = torch.randint(0, T, (demo_images.size(0),), device=device)
demo_x_t = q_sample(demo_images, demo_t, demo_noise)

model.eval()
with torch.no_grad():
    demo_pred = model(demo_x_t, demo_t)

demo_mse = F.mse_loss(demo_pred, demo_noise)
idx = 0
single_mse = F.mse_loss(demo_pred[idx], demo_noise[idx])

print("Single-batch training demo")
print(f"  Batch shape:        {tuple(demo_images.shape)}")
print(f"  Example index:      {idx}")
print(f"  Sampled t:          {demo_t[idx].item()}")
print(f"  alpha_bar at t:     {alpha_bars[demo_t[idx]].item():.6f}")
print(f"  MSE (one example):  {single_mse.item():.6f}")
print(f"  MSE (full batch):   {demo_mse.item():.6f}")
print("  Goal during training: drive batch MSE down across random timesteps.")


### Step 4: Full training loop

Each step picks a random `t` per image, adds noise, and regresses the network output toward the sampled `epsilon`. Three epochs over 8k images is enough to see the loss fall and produce rough digit shapes at sample time.


In [ ]:
loss_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for images, _ in loader:
        images = images.to(device)
        noise = torch.randn_like(images)
        t = torch.randint(0, T, (images.size(0),), device=device)
        x_t = q_sample(images, t, noise)

        optimizer.zero_grad()
        pred_noise = model(x_t, t)
        loss = F.mse_loss(pred_noise, noise)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg = epoch_loss / len(loader)
    loss_history.append(avg)
    print(f"Epoch {epoch}/{EPOCHS}, average noise MSE: {avg:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("DDPM noise-prediction loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Step 5: Reverse diffusion sampling

Sampling reverses the forward chain: begin with `x_T ~ N(0, I)` and for `t = T-1 ... 0` predict noise, estimate `x_0`, and compute the posterior mean for `x_{t-1}`. We generate 16 images and print their pixel statistics.


In [ ]:
@torch.no_grad()
def sample_ddpm(n: int = 16) -> torch.Tensor:
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)
    for step in reversed(range(T)):
        t = torch.full((n,), step, device=device, dtype=torch.long)
        pred_noise = model(x, t)
        ab = alpha_bars[step].to(device)
        ab_prev = alpha_bars[step - 1].to(device) if step > 0 else torch.tensor(1.0, device=device)
        beta = betas[step].to(device)
        alpha = alphas[step].to(device)

        x0_pred = (x - torch.sqrt(1.0 - ab) * pred_noise) / torch.sqrt(ab)
        x0_pred = x0_pred.clamp(-1.0, 1.0)

        if step > 0:
            coef1 = torch.sqrt(ab_prev) * beta / (1.0 - ab)
            coef2 = torch.sqrt(alpha) * (1.0 - ab_prev) / (1.0 - ab)
            mean = coef1 * x0_pred + coef2 * x
            noise = torch.randn_like(x)
            x = mean + torch.sqrt(beta) * noise
        else:
            x = x0_pred
    return x


samples = sample_ddpm(16).cpu()
print("Generated sample statistics (16 images)")
print(f"  Shape:      {tuple(samples.shape)}")
print(f"  Mean:       {samples.mean():.4f}")
print(f"  Std:        {samples.std():.4f}")
print(f"  Min / max:  {samples.min():.3f} / {samples.max():.3f}")
print("  After more epochs, means stay near 0 but digit structure becomes sharper.")

grid = make_grid(samples, nrow=4, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(5, 5))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("DDPM samples (16 images)")
plt.axis("off")
plt.tight_layout()
plt.show()
